# 02 — Temporal transition analysis

This notebook builds on the niche-level representations from `01_explore_and_embed`
to analyze how spatial organization changes across disease stages.

**Part A**: Descriptive analysis — trajectory visualization, transition vectors,
cross-model comparisons. No model training needed, immediate scientific insight.

**Part B**: Transition model — learn to predict state transitions and identify
what spatial features drive them.

**Part C**: Factor attribution — what distinguishes niches that transition toward
remission vs. chronicity?

In [ ]:
import sys
sys.path.insert(0, "../src")

%load_ext autoreload
%autoreload 2

import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path
from scipy.spatial.distance import cosine, cdist
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn

from temporal_foundation.config import (
    MODELS, SHARED_STAGES, STAGE_KEY, MODEL_KEY, REGION_KEY, SAMPLE_KEY,
    CELL_TYPE_KEYS, EmbeddingConfig, AggregationConfig,
    get_all_stages, get_transition_pairs, model_short_name,
)
from temporal_foundation.data import load_adata, split_by_sample, build_sample_table
from temporal_foundation.embeddings import load_embeddings
from temporal_foundation.aggregation import (
    aggregate_all_samples, aggregate_sample, summarize_representations,
    NicheRepresentation,
)

## Load niche representations

Load the embedded data from notebook 01. If you saved the embedded samples,
load from disk. Otherwise, recompute from the raw data.

In [ ]:
# Option A: Load pre-computed embedded samples from disk
EMBEDDED_DIR = Path("../data/embedded")

embed_config = EmbeddingConfig()
agg_config = AggregationConfig(
    method="spatial_domains",
    domain_key=embed_config.domain_key,
    embedding_key=embed_config.latent_key,
    include_composition=True,
    cell_type_key="anno_L2",
)

if EMBEDDED_DIR.exists() and list(EMBEDDED_DIR.glob("*.h5ad")):
    samples = load_embeddings(EMBEDDED_DIR, embed_config)
else:
    # Option B: Load raw data and recompute (slower)
    print("No pre-computed embeddings found. Loading raw data...")
    adata = load_adata()
    samples = split_by_sample(adata)
    from temporal_foundation.embeddings import compute_novae_batch
    samples = compute_novae_batch(samples, embed_config)

# Aggregate to niche level
representations = aggregate_all_samples(samples, agg_config)
summary = summarize_representations(representations)
print(f"\n{len(representations)} samples, {summary['n_niches'].sum()} total niches")
summary.head()

---
# Part A: Descriptive transition analysis

No model training — just analyzing the structure of the embedding space across
disease stages to understand how spatial organization evolves.

## A1. Sample-level trajectory in embedding space

Compute a single embedding per sample (mean of niche embeddings), then
visualize the disease trajectory as a path through this space.

In [ ]:
# Build sample-level representations: one embedding vector per sample
sample_data = []
for rep in representations:
    sample_data.append({
        "sample_id": rep.sample_id,
        "model": rep.model,
        "stage": rep.stage,
        "region": rep.region,
        "n_niches": rep.n_niches,
        "clinical_score": rep.clinical_score,
        "embedding": rep.embeddings.mean(axis=0),  # mean over niches
    })

sample_df = pd.DataFrame(sample_data)
sample_embeddings = np.stack(sample_df["embedding"].values)

print(f"Samples: {len(sample_df)}")
print(f"Embedding dim: {sample_embeddings.shape[1]}")
print(f"\nSamples per model:")
print(sample_df.groupby("model").size())

In [ ]:
# PCA -> UMAP of sample-level embeddings
import umap

scaler = StandardScaler()
emb_scaled = scaler.fit_transform(sample_embeddings)

pca = PCA(n_components=min(20, emb_scaled.shape[1]))
emb_pca = pca.fit_transform(emb_scaled)
print(f"PCA: {pca.explained_variance_ratio_[:5].sum()*100:.1f}% variance in first 5 PCs")

reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.3)
emb_umap = reducer.fit_transform(emb_pca)

sample_df["umap_0"] = emb_umap[:, 0]
sample_df["umap_1"] = emb_umap[:, 1]

In [ ]:
# Plot trajectories: connect stage means with arrows
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
all_stages = get_all_stages()

for ax, model_name in zip(axes, ["MOG", "PLP"]):
    stages = all_stages[model_name]
    model_mask = sample_df["model"] == model_name
    
    # Plot all samples as scatter points
    for other_model in ["MOG", "PLP"]:
        other_mask = sample_df["model"] == other_model
        alpha = 0.8 if other_model == model_name else 0.15
        color = None if other_model == model_name else "lightgrey"
        
        if other_model == model_name:
            # Color by stage order
            stage_order = {s: i for i, s in enumerate(stages)}
            colors = sample_df.loc[other_mask, "stage"].map(stage_order)
            scatter = ax.scatter(
                sample_df.loc[other_mask, "umap_0"],
                sample_df.loc[other_mask, "umap_1"],
                c=colors, cmap="Spectral", s=60, alpha=alpha,
                edgecolors="k", linewidths=0.5, zorder=3
            )
        else:
            ax.scatter(
                sample_df.loc[other_mask, "umap_0"],
                sample_df.loc[other_mask, "umap_1"],
                c=color, s=20, alpha=alpha, zorder=1
            )
    
    # Compute stage centroids and draw trajectory arrows
    centroids = {}
    for stage in stages:
        mask = model_mask & (sample_df["stage"] == stage)
        if mask.any():
            centroids[stage] = sample_df.loc[mask, ["umap_0", "umap_1"]].mean().values
    
    for i in range(len(stages) - 1):
        s1, s2 = stages[i], stages[i + 1]
        if s1 in centroids and s2 in centroids:
            ax.annotate(
                "", xy=centroids[s2], xytext=centroids[s1],
                arrowprops=dict(arrowstyle="->", color="black", lw=2),
                zorder=5
            )
    
    # Label centroids
    for stage, pos in centroids.items():
        ax.annotate(stage, pos, fontsize=7, fontweight="bold", ha="center",
                    va="bottom", xytext=(0, 8), textcoords="offset points",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.8),
                    zorder=6)
    
    ax.set_title(f"{model_name} trajectory", fontsize=13)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")

plt.suptitle("Disease trajectories in niche embedding space", fontsize=14)
plt.tight_layout()
plt.show()

## A2. Transition vectors

For each consecutive stage pair, compute the **transition vector** — the direction
of change in embedding space. This tells us how the spatial organization shifts
at each step of the disease.

In [ ]:
# Compute mean embedding per (model, stage)
stage_means = {}
stage_samples = {}
for _, row in sample_df.iterrows():
    key = (row["model"], row["stage"])
    if key not in stage_means:
        stage_means[key] = []
        stage_samples[key] = 0
    stage_means[key].append(row["embedding"])
    stage_samples[key] += 1

for key in stage_means:
    stage_means[key] = np.mean(stage_means[key], axis=0)

# Compute transition vectors for each model
transitions = {}
for model_name in ["MOG", "PLP"]:
    pairs = get_transition_pairs(model_name)
    for from_s, to_s in pairs:
        k1, k2 = (model_name, from_s), (model_name, to_s)
        if k1 in stage_means and k2 in stage_means:
            vec = stage_means[k2] - stage_means[k1]
            transitions[(model_name, from_s, to_s)] = {
                "vector": vec,
                "magnitude": np.linalg.norm(vec),
                "cosine_dist": cosine(stage_means[k1], stage_means[k2]),
                "n_from": stage_samples[k1],
                "n_to": stage_samples[k2],
            }

# Print transition magnitudes
for model_name in ["MOG", "PLP"]:
    print(f"\n{model_name} transition magnitudes:")
    pairs = get_transition_pairs(model_name)
    for from_s, to_s in pairs:
        key = (model_name, from_s, to_s)
        if key in transitions:
            t = transitions[key]
            print(f"  {from_s:15s} -> {to_s:15s}  "
                  f"magnitude={t['magnitude']:.4f}  "
                  f"cosine={t['cosine_dist']:.4f}  "
                  f"(n={t['n_from']}->{t['n_to']})")

### Are PLP relapses using the same spatial programs?

Compare the transition vectors for PEAK1->REMISSION1 vs PEAK2->REMISSION2.
If they point in the same direction, remission uses the same spatial reorganization
each time. If different, the tissue finds a different path to recovery.

In [ ]:
# Compare transition vector similarity
comparisons = [
    # Same transition type across PLP cycles
    ("PLP", "PEAK1", "REMISSION1", "PLP", "PEAK2", "REMISSION2",
     "Peak->Remission: cycle 1 vs cycle 2"),
    ("PLP", "REMISSION1", "PEAK2", "PLP", "REMISSION2", "PEAK3",
     "Remission->Peak: relapse 1 vs relapse 2"),
    # Disease onset vs relapse
    ("PLP", "ONSET1", "ONSET2", "PLP", "REMISSION1", "PEAK2",
     "First onset vs first relapse"),
    # Forward vs reverse transitions
    ("PLP", "PEAK1", "REMISSION1", "PLP", "REMISSION1", "PEAK2",
     "Peak->Remission vs Remission->Peak (should be opposite)"),
]

print("Transition vector similarity (cosine of angle between vectors):")
print("  1.0 = identical direction, 0.0 = orthogonal, -1.0 = opposite\n")

for m1, f1, t1, m2, f2, t2, label in comparisons:
    k1 = (m1, f1, t1)
    k2 = (m2, f2, t2)
    if k1 in transitions and k2 in transitions:
        v1 = transitions[k1]["vector"]
        v2 = transitions[k2]["vector"]
        similarity = 1 - cosine(v1, v2)  # cosine similarity
        print(f"  {f1}->{t1} vs {f2}->{t2}")
        print(f"    similarity = {similarity:+.4f}  ({label})")
        print()

## A3. Cell type composition changes across transitions

How does the cell type makeup of niches change at each transition?
This is directly interpretable — no embedding required.

In [ ]:
# Compute mean cell type composition per (model, stage)
# Only works if representations have composition data
has_composition = any(r.composition is not None for r in representations)

if has_composition:
    cell_type_names = representations[0].cell_type_names
    
    stage_compositions = {}
    for rep in representations:
        if rep.composition is None:
            continue
        key = (rep.model, rep.stage)
        if key not in stage_compositions:
            stage_compositions[key] = []
        # Weighted mean by cell count
        weights = rep.cell_counts / rep.cell_counts.sum()
        weighted_comp = (rep.composition * weights[:, None]).sum(axis=0)
        stage_compositions[key].append(weighted_comp)
    
    for key in stage_compositions:
        stage_compositions[key] = np.mean(stage_compositions[key], axis=0)
    
    # Plot composition across PLP trajectory
    plp_stages = get_all_stages()["PLP"]
    plp_comps = []
    plp_stage_labels = []
    for stage in plp_stages:
        key = ("PLP", stage)
        if key in stage_compositions:
            plp_comps.append(stage_compositions[key])
            plp_stage_labels.append(stage)
    
    plp_comps = np.stack(plp_comps)
    
    fig, ax = plt.subplots(figsize=(14, 6))
    bottom = np.zeros(len(plp_stage_labels))
    colors = plt.cm.tab20(np.linspace(0, 1, len(cell_type_names)))
    
    for i, ct in enumerate(cell_type_names):
        ax.bar(plp_stage_labels, plp_comps[:, i], bottom=bottom,
               label=ct, color=colors[i])
        bottom += plp_comps[:, i]
    
    ax.set_ylabel("Proportion")
    ax.set_title("PLP cell type composition across disease stages")
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No composition data available — rerun aggregation with include_composition=True")

In [ ]:
# Composition CHANGE at each transition — what cell types shift most?
if has_composition:
    print("Cell type changes at each PLP transition (positive = increase):\n")
    
    plp_pairs = get_transition_pairs("PLP")
    change_data = []
    
    for from_s, to_s in plp_pairs:
        k1, k2 = ("PLP", from_s), ("PLP", to_s)
        if k1 in stage_compositions and k2 in stage_compositions:
            delta = stage_compositions[k2] - stage_compositions[k1]
            print(f"{from_s} -> {to_s}:")
            # Show top 5 increasing and decreasing
            sorted_idx = np.argsort(delta)
            for idx in sorted_idx[:3]:  # top decreasing
                print(f"  {cell_type_names[idx]:30s} {delta[idx]:+.4f}")
            print(f"  ...")
            for idx in sorted_idx[-3:]:  # top increasing
                print(f"  {cell_type_names[idx]:30s} {delta[idx]:+.4f}")
            print()
            
            for i, ct in enumerate(cell_type_names):
                change_data.append({
                    "from": from_s, "to": to_s,
                    "cell_type": ct, "delta": delta[i],
                })
    
    change_df = pd.DataFrame(change_data)

## A4. Cross-model comparison at PEAK1

PEAK1 has samples from both MOG and PLP. MOG goes chronic, PLP goes to remission.
What's spatially different between them at this shared disease state?

In [ ]:
# Compare MOG vs PLP at PEAK1
for shared_stage in SHARED_STAGES:
    mog_reps = [r for r in representations if r.model == "MOG" and r.stage == shared_stage]
    plp_reps = [r for r in representations if r.model == "PLP" and r.stage == shared_stage]
    
    if not mog_reps or not plp_reps:
        print(f"No data for {shared_stage} in one of the models")
        continue
    
    print(f"=== {shared_stage}: MOG ({len(mog_reps)} samples) vs PLP ({len(plp_reps)} samples) ===\n")
    
    # Embedding distance
    mog_mean = np.mean([r.embeddings.mean(axis=0) for r in mog_reps], axis=0)
    plp_mean = np.mean([r.embeddings.mean(axis=0) for r in plp_reps], axis=0)
    print(f"Cosine distance between MOG and PLP at {shared_stage}: {cosine(mog_mean, plp_mean):.4f}")
    
    # Composition differences
    if has_composition and mog_reps[0].composition is not None:
        mog_comp = np.mean([
            (r.composition * (r.cell_counts / r.cell_counts.sum())[:, None]).sum(axis=0)
            for r in mog_reps
        ], axis=0)
        plp_comp = np.mean([
            (r.composition * (r.cell_counts / r.cell_counts.sum())[:, None]).sum(axis=0)
            for r in plp_reps
        ], axis=0)
        
        delta = plp_comp - mog_comp  # positive = more in PLP
        sorted_idx = np.argsort(np.abs(delta))[::-1]
        
        print(f"\nBiggest composition differences (PLP minus MOG):")
        print(f"  Positive = more in PLP (which will remit)")
        print(f"  Negative = more in MOG (which goes chronic)\n")
        for idx in sorted_idx[:10]:
            marker = "***" if abs(delta[idx]) > 0.02 else ""
            print(f"  {cell_type_names[idx]:30s} {delta[idx]:+.4f} {marker}")

---
# Part B: Transition model

Learn to predict the spatial state at the next stage from the current state.
The model operates on sample-level embeddings (mean of niche embeddings).

Given the small sample size (~3 per stage), we start with simple models and
increase complexity only if warranted:
1. **Linear transition model** — learn a matrix that maps stage t to stage t+1
2. **Stage-conditioned model** — different transition per stage type (peak->remission vs remission->peak)
3. **Attention-based model** — attend over niche embeddings to identify which niches drive the transition

## B1. Build training data from transition pairs

Each training example is: (sample embedding at stage t) -> (mean embedding at stage t+1).
Since we don't track individual animals across stages, we pair each sample at stage t
with the mean of all samples at stage t+1.

In [ ]:
# Build transition pairs for training
# For each sample at stage t, the target is the centroid of stage t+1
# We also encode the transition type as metadata

train_pairs = []
for model_name in ["MOG", "PLP"]:
    pairs = get_transition_pairs(model_name)
    for from_stage, to_stage in pairs:
        k_to = (model_name, to_stage)
        if k_to not in stage_means:
            continue
        
        target = stage_means[k_to]
        
        # Each sample at from_stage is a training example
        from_mask = (sample_df["model"] == model_name) & (sample_df["stage"] == from_stage)
        for idx in sample_df[from_mask].index:
            source_emb = sample_df.loc[idx, "embedding"]
            train_pairs.append({
                "model": model_name,
                "from_stage": from_stage,
                "to_stage": to_stage,
                "sample_id": sample_df.loc[idx, "sample_id"],
                "region": sample_df.loc[idx, "region"],
                "source": source_emb,
                "target": target,
                # Transition type: toward disease or toward recovery
                "direction": "recovery" if "REMISSION" in to_stage or to_stage == "LONG" else "disease",
            })

train_df = pd.DataFrame(train_pairs)
X_train = np.stack(train_df["source"].values)
Y_train = np.stack(train_df["target"].values)

print(f"Training pairs: {len(train_df)}")
print(f"Input dim: {X_train.shape[1]}")
print(f"\nPairs per model:")
print(train_df.groupby("model").size())
print(f"\nPairs per transition direction:")
print(train_df.groupby("direction").size())
print(f"\nPairs per transition:")
print(train_df.groupby(["model", "from_stage", "to_stage"]).size())

## B2. Linear transition model

The simplest model: learn a linear map from current state to next state.
If this works, it means transitions are approximately linear in embedding space
and we can directly interpret the weight matrix.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import LeaveOneOut, cross_val_predict

# Ridge regression: predict next-stage embedding from current-stage embedding
# Use ridge (L2 regularization) to handle the high-dimensional, low-sample regime
alphas = [0.01, 0.1, 1.0, 10.0, 100.0]
best_score = -np.inf
best_alpha = None

for alpha in alphas:
    model = Ridge(alpha=alpha)
    # Leave-one-out cross-validation
    loo = LeaveOneOut()
    scores = []
    for train_idx, test_idx in loo.split(X_train):
        model.fit(X_train[train_idx], Y_train[train_idx])
        score = model.score(X_train[test_idx], Y_train[test_idx])
        scores.append(score)
    mean_score = np.mean(scores)
    if mean_score > best_score:
        best_score = mean_score
        best_alpha = alpha

print(f"Best alpha: {best_alpha} (LOO R² = {best_score:.4f})")

# Fit final model
linear_model = Ridge(alpha=best_alpha)
linear_model.fit(X_train, Y_train)
train_score = linear_model.score(X_train, Y_train)
print(f"Train R²: {train_score:.4f}")

# LOO predictions for evaluation
Y_pred_loo = cross_val_predict(Ridge(alpha=best_alpha), X_train, Y_train, cv=LeaveOneOut())

In [ ]:
# Evaluate: for each sample, is the predicted next-stage closer to the correct
# next-stage than to other stages?
correct = 0
total = 0

for i, (_, row) in enumerate(train_df.iterrows()):
    pred = Y_pred_loo[i]
    target_stage = row["to_stage"]
    model_name = row["model"]
    
    # Distance to correct next stage
    dist_correct = cosine(pred, stage_means[(model_name, target_stage)])
    
    # Distance to all other stages for this model
    other_stages = [s for s in get_all_stages()[model_name] if s != target_stage]
    closer_count = 0
    for other_s in other_stages:
        if (model_name, other_s) in stage_means:
            dist_other = cosine(pred, stage_means[(model_name, other_s)])
            if dist_correct < dist_other:
                closer_count += 1
    
    # Correct if predicted embedding is closer to target than to most other stages
    if closer_count >= len(other_stages) * 0.5:
        correct += 1
    total += 1

print(f"Transition prediction accuracy: {correct}/{total} ({100*correct/total:.1f}%)")
print(f"(Predicted embedding closer to correct next stage than to most other stages)")

## B3. Attention-based niche transition model

This model operates on **niche-level** embeddings (not sample means) to identify
which specific spatial neighborhoods are most informative about the transition.

Architecture:
1. Attention pooling over niches within a sample -> sample representation
2. Linear transition prediction -> predicted next-stage embedding
3. The attention weights tell us which niches the model considers important

In [ ]:
class NicheAttentionTransitionModel(nn.Module):
    """Attention-based transition model operating on niche-level embeddings.
    
    For each sample (set of niche embeddings):
    1. Compute attention weights over niches
    2. Weighted pool to get sample representation
    3. Predict next-stage embedding via linear layer
    
    The attention weights identify which niches are most important for
    predicting the transition.
    """
    
    def __init__(self, embed_dim: int, hidden_dim: int = 64, dropout: float = 0.1):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )
        self.transition = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim),
        )
    
    def forward(self, niche_embeddings: torch.Tensor):
        """
        Args:
            niche_embeddings: (n_niches, embed_dim) for a single sample
        Returns:
            predicted: (embed_dim,) predicted next-stage embedding
            attention_weights: (n_niches,) attention over niches
        """
        # Attention
        attn_logits = self.attention(niche_embeddings).squeeze(-1)  # (n_niches,)
        attn_weights = torch.softmax(attn_logits, dim=0)           # (n_niches,)
        
        # Weighted pool
        sample_repr = (attn_weights.unsqueeze(-1) * niche_embeddings).sum(dim=0)  # (embed_dim,)
        
        # Predict transition
        predicted = self.transition(sample_repr)  # (embed_dim,)
        
        return predicted, attn_weights


print(f"Model parameters: {sum(p.numel() for p in NicheAttentionTransitionModel(64).parameters()):,}")

In [ ]:
# Build niche-level training data
# Each example: (niche_embeddings for one sample, target stage mean embedding)
niche_train_data = []

rep_lookup = {r.sample_id: r for r in representations}

for _, row in train_df.iterrows():
    sample_id = row["sample_id"]
    if sample_id not in rep_lookup:
        continue
    rep = rep_lookup[sample_id]
    target = row["target"]
    
    niche_train_data.append({
        "sample_id": sample_id,
        "model": row["model"],
        "from_stage": row["from_stage"],
        "to_stage": row["to_stage"],
        "niche_embeddings": torch.tensor(rep.embeddings, dtype=torch.float32),
        "target": torch.tensor(target, dtype=torch.float32),
        "niche_centroids": rep.centroids,
        "niche_cell_counts": rep.cell_counts,
    })

print(f"Niche-level training examples: {len(niche_train_data)}")
if niche_train_data:
    print(f"Niche embedding dim: {niche_train_data[0]['niche_embeddings'].shape[1]}")

In [ ]:
# Train the attention model
embed_dim = niche_train_data[0]["niche_embeddings"].shape[1]

attn_model = NicheAttentionTransitionModel(embed_dim=embed_dim, hidden_dim=64, dropout=0.1)
optimizer = torch.optim.Adam(attn_model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.MSELoss()

# Training loop
n_epochs = 500
losses = []

attn_model.train()
for epoch in range(n_epochs):
    epoch_loss = 0
    for example in niche_train_data:
        optimizer.zero_grad()
        pred, _ = attn_model(example["niche_embeddings"])
        loss = criterion(pred, example["target"])
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    epoch_loss /= len(niche_train_data)
    losses.append(epoch_loss)
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}/{n_epochs}  loss={epoch_loss:.6f}")

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(losses)
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("Attention model training")
plt.tight_layout()
plt.show()

---
# Part C: Factor attribution

Extract attention weights from the trained model to identify which spatial
niches are most important for predicting transitions. Map these back to
spatial coordinates and cell type compositions to understand **what** is
driving each transition.

In [ ]:
# Extract attention weights for all training examples
attn_model.eval()
attention_results = []

with torch.no_grad():
    for example in niche_train_data:
        pred, attn_weights = attn_model(example["niche_embeddings"])
        attention_results.append({
            "sample_id": example["sample_id"],
            "model": example["model"],
            "from_stage": example["from_stage"],
            "to_stage": example["to_stage"],
            "attention_weights": attn_weights.numpy(),
            "niche_centroids": example["niche_centroids"],
            "niche_cell_counts": example["niche_cell_counts"],
        })

print(f"Extracted attention for {len(attention_results)} samples")

### Spatial attention maps

For key transitions, plot which spatial niches receive the most attention.
High-attention niches are the ones the model considers most informative about
where the tissue is headed.

In [ ]:
# Plot spatial attention maps for key transitions
key_transitions = [
    ("PLP", "PEAK1", "REMISSION1"),      # What predicts first remission?
    ("PLP", "REMISSION1", "PEAK2"),       # What predicts first relapse?
    ("PLP", "PEAK2", "REMISSION2"),       # What predicts second remission?
    ("MOG", "PEAK1", "LONG"),             # What predicts chronicity?
]

for model_name, from_s, to_s in key_transitions:
    examples = [r for r in attention_results
                if r["model"] == model_name and r["from_stage"] == from_s and r["to_stage"] == to_s]
    
    if not examples:
        continue
    
    n_examples = min(len(examples), 3)
    fig, axes = plt.subplots(1, n_examples, figsize=(6 * n_examples, 5))
    if n_examples == 1:
        axes = [axes]
    
    for ax, ex in zip(axes, examples[:n_examples]):
        centroids = ex["niche_centroids"]
        attn = ex["attention_weights"]
        
        scatter = ax.scatter(
            centroids[:, 0], centroids[:, 1],
            c=attn, s=ex["niche_cell_counts"] * 0.5,
            cmap="Reds", vmin=0, alpha=0.8, edgecolors="k", linewidths=0.3
        )
        plt.colorbar(scatter, ax=ax, label="Attention weight", shrink=0.8)
        ax.set_title(f"{ex['sample_id']}", fontsize=10)
        ax.set_aspect("equal")
        ax.axis("off")
    
    fig.suptitle(f"{model_name}: {from_s} -> {to_s}\n(high attention = niche most informative about transition)",
                 fontsize=12)
    plt.tight_layout()
    plt.show()

### What cell types are in high-attention niches?

For each transition, compare the cell type composition of high-attention vs
low-attention niches. This answers: what cell types are most associated with
the transition signal?

In [ ]:
# Compare cell type composition: high-attention vs low-attention niches
if has_composition:
    for model_name, from_s, to_s in key_transitions:
        examples_attn = [r for r in attention_results
                         if r["model"] == model_name and r["from_stage"] == from_s and r["to_stage"] == to_s]
        examples_rep = [rep_lookup[r["sample_id"]] for r in examples_attn if r["sample_id"] in rep_lookup]
        
        if not examples_attn or not examples_rep or examples_rep[0].composition is None:
            continue
        
        # Pool across all samples for this transition
        all_attn = np.concatenate([r["attention_weights"] for r in examples_attn])
        all_comp = np.vstack([r.composition for r in examples_rep])
        
        # Split into high vs low attention (above/below median)
        median_attn = np.median(all_attn)
        high_mask = all_attn > median_attn
        low_mask = ~high_mask
        
        high_comp = all_comp[high_mask].mean(axis=0)
        low_comp = all_comp[low_mask].mean(axis=0)
        delta = high_comp - low_comp
        
        sorted_idx = np.argsort(np.abs(delta))[::-1]
        
        print(f"\n{model_name}: {from_s} -> {to_s}")
        print(f"  High-attention niches vs low-attention niches:")
        print(f"  (positive = enriched in transition-driving niches)\n")
        for idx in sorted_idx[:8]:
            bar = "+" * int(abs(delta[idx]) * 200) if delta[idx] > 0 else "-" * int(abs(delta[idx]) * 200)
            print(f"  {cell_type_names[idx]:30s} {delta[idx]:+.4f}  {bar}")

## Summary

**Part A** gives us descriptive insight:
- How disease trajectories move through the niche embedding space
- Whether PLP relapses reuse the same spatial transition programs
- What cell types shift at each transition
- How MOG and PLP differ at their shared PEAK1 stage

**Part B** tests whether transitions are predictable:
- Linear model: are transitions approximately linear in embedding space?
- Attention model: which niches carry the most transition signal?

**Part C** provides biological interpretation:
- Spatial attention maps show *where* in the tissue the transition signal lives
- Cell type enrichment in high-attention niches shows *what* is driving the signal
- Comparing PLP PEAK1->REMISSION1 vs MOG PEAK1->LONG identifies candidate
  mechanisms of remission vs. chronicity